# Final results figures

Starting figure set: a global Mollweide location map, followed by five matched views of **example 5, unit 35**. Run with the `river-hierarchy-rivgraph` environment (GeoPandas, Rasterio, Matplotlib, pandas, NumPy). No Cartopy download is needed.

**Provenance:** the requested source panel is **2020Q4**, but `prepare_summary.json` records **2020Q1** as the input to the existing prepared/cleaned masks. These panels therefore show the requested assets, not an exact same-quarter processing sequence. Existing analysis products are read only.

“Centerline” defaults to the complete channel network; change `CENTERLINE_KIND` to `main` for RivGraph's single main-river centerline. The last panel adds the actual collapsed child link (identified by its lineage) and the filled island footprint, over the original cleaned mask. All five detail panels use the same projected extent and raster grid.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Ellipse
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'rivgraph_centerline').is_dir())
DATA_ROOT = Path('/Volumes/PhD/river_hierarchy/data')
# Local Natural Earth 1:10 million assets; override if moved or copied into the repository.
BASEMAP_DIR = Path('/Users/6256481/Desktop/PhD_icloud/projecten/river_hierarchy/paper/figuren/methode/10m_physical')
OUTPUT_DIR = ROOT / 'final_results' / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CENTERLINE_KIND = 'network'  # 'network' or 'main'
SAVE_FIGURES = True
SAVE_PROVENANCE = False  # Set True only when companion source JSON is needed.
DPI = 300
PADDING_FRACTION = 0.25
WATER_COLOR = 'black'
LAND_COLOR = 'white'
NETWORK_COLOR = '#2764a0'
COLLAPSED_COLOR = 'red'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                     'savefig.facecolor': 'white', 'figure.facecolor': 'white'})

def finish(fig, name):
    if SAVE_FIGURES:
        fig.savefig(OUTPUT_DIR / f'{name}.png', dpi=DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 1.1 Global example locations
Natural Earth 10m land and ocean polygons in Mollweide projection; ocean is white and example locations are red. Each dot is the projected bounding-box center of one example in the final 17-example dataset.

In [ ]:
LOCATIONS_PATH = ROOT / 'SWORD_gauge_match/outputs/hierarchy_examples_final_example_bboxes.gpkg'
MOLLWEIDE = '+proj=moll +lon_0=0 +datum=WGS84 +units=m +no_defs'
locations = gpd.read_file(LOCATIONS_PATH)
locations = locations = locations[~locations['example_id'].isin([10,11])]

assert locations['example_id'].is_unique
points = locations.to_crs('EPSG:6933').geometry.centroid.to_crs(MOLLWEIDE)
land_path = BASEMAP_DIR / 'ne_10m_land.shp'
ocean_path = BASEMAP_DIR / 'ne_10m_ocean.shp'
for path in (land_path, ocean_path):
    if not path.exists():
        raise FileNotFoundError(f'Natural Earth layer missing: {path}. Set BASEMAP_DIR above.')
land = gpd.read_file(land_path).to_crs(MOLLWEIDE)
ocean = gpd.read_file(ocean_path).to_crs(MOLLWEIDE)
fig, ax = plt.subplots(figsize=(11, 5.8))
world = Ellipse((0, 0), width=36080191.4, height=18040095.7,
                facecolor='white', edgecolor='white', linewidth=0.6, zorder=0)
ax.add_patch(world)
ocean.plot(ax=ax, facecolor='#E0E0E0', edgecolor='none', zorder=1)
land.plot(ax=ax, facecolor='black', edgecolor='none', zorder=2)
for collection in ax.collections:
    collection.set_clip_path(world)
ax.scatter(points.x, points.y, color='red', s=28, edgecolors='white',
           linewidths=0.45, zorder=3)
ax.set(xlim=(-18150000, 18150000), ylim=(-9100000, 9100000), aspect='equal')
ax.set_axis_off()
finish(fig, '1_1_global_locations')
print(f'{len(locations)} example locations; Natural Earth 10m land + white ocean.')


# 2. mask operations and centerline pipeline

## 0. Detail inputs and shared unit 35 extent
The crop is derived from unit 35's base links and its collapse footprint, with 25% padding. Raster panels use nearest-neighbor reprojection onto one grid in the cleaned mask CRS. Source values retain their original scale; binary masks use 0 = land, positive = water. Missing pixels stay transparent.

In [ ]:
RUN = ROOT / 'rivgraph_centerline/outputs/runs/05'
EXPERIMENT = ROOT / 'network_variants/output/05_indep__anchored'
BASE = EXPERIMENT / 'states/S000_base'
STATE = EXPERIMENT / 'states/S035_unit_35'
SOURCE = DATA_ROOT / 'water_masks/example_5_2020Q4_max_water_uint8.tif'
PREPARED = RUN / 'masks_prepared/05_binary_projected.tif'
CLEANED = RUN / 'masks_cleaned/05_cleaned.tif'
NETWORK = RUN / 'rivgraph/05_links.gpkg'
MAIN = RUN / 'rivgraph/05_centerline.gpkg'
BASE_LINKS = BASE / 'variant/directed/05__S000_base_directed_links.gpkg'
CHILD_LINKS = STATE / 'variant/directed/05__S035_unit_35_directed_links.gpkg'
EDITS = STATE / 'variant/mask/collapse_edit_geometries.gpkg'
MEMBERSHIP = BASE / 'variant/summary/unit_link_membership.csv'
with rasterio.open(CLEANED) as src:
    CRS = src.crs
    pixel_size = abs(src.res[0])
membership = pd.read_csv(MEMBERSHIP)
unit_ids = membership.loc[membership.unit_id.eq(35), 'id_link']
base_links = gpd.read_file(BASE_LINKS).to_crs(CRS)
unit_links = base_links.loc[base_links.id_link.isin(unit_ids)]
assert len(unit_links), 'Unit 35 is missing from the base network.'
edits = gpd.read_file(EDITS).to_crs(CRS)
fills = edits.loc[edits.action.eq('add')]
child_links = gpd.read_file(CHILD_LINKS).to_crs(CRS)
collapsed = child_links.loc[child_links.lineage_type.eq('collapsed_many_to_one')]
assert len(collapsed), 'No collapsed child link found.'
assert set(edits.collapsed_unit_ids.astype(str)) == {'35'}
assert CENTERLINE_KIND in ('network', 'main')
centerlines = gpd.read_file(NETWORK if CENTERLINE_KIND == 'network' else MAIN).to_crs(CRS)
geometry = gpd.GeoSeries([*unit_links.geometry, *edits.geometry, *collapsed.geometry], crs=CRS)
x0, y0, x1, y1 = geometry.total_bounds
pad = max(x1-x0, y1-y0) * PADDING_FRACTION
BOUNDS = (x0-pad, y0-pad, x1+pad, y1+pad)
x0, y0, x1, y1 = BOUNDS
width = int(np.ceil((x1-x0) / pixel_size))
height = int(np.ceil((y1-y0) / pixel_size))
TRANSFORM = from_bounds(*BOUNDS, width, height)

def read_panel(path):
    with rasterio.open(path) as src:
        with WarpedVRT(src, crs=CRS, transform=TRANSFORM, width=width, height=height,
                       resampling=Resampling.nearest) as vrt:
            return vrt.read(1, masked=True)

source_image = read_panel(SOURCE)
prepared_image = read_panel(PREPARED)
cleaned_image = read_panel(CLEANED)
assert all(image.count() for image in (source_image, prepared_image, cleaned_image))

def detail_panel(title, values, binary=True, bounds=None, scale_position="right"):
    x0, y0, x1, y1 = BOUNDS if bounds is None else bounds
    fig, ax = plt.subplots(figsize=(7, 7))
    kwargs = dict(extent=(x0, x1, y0, y1), origin='upper', interpolation='nearest')
    if binary:
        ax.imshow(np.ma.array(values > 0, mask=np.ma.getmaskarray(values)),
                  cmap=ListedColormap([LAND_COLOR, WATER_COLOR]), vmin=0, vmax=1, **kwargs)
    else:
        ax.imshow(values, cmap='gray_r', vmin=0, vmax=80, **kwargs)
    ax.set(xlim=(x0, x1), ylim=(y0, y1), aspect='equal', title=title)
    ax.set_axis_off()
    # Fixed 500 m scale bar in the shared projected CRS.
    sx = x0 + (x1-x0)*0.06 if scale_position == "left" else x1 - (x1-x0)*0.06 - 500
    sy = y0 + (y1-y0)*0.07
    # ax.plot([sx, sx+500], [sy, sy], color='black', linewidth=2)
    # ax.text(sx+250, sy+(y1-y0)*0.018, '500 m', ha='center', fontsize=9)
    return fig, ax

provenance = {'source': str(SOURCE), 'prepared': str(PREPARED), 'cleaned': str(CLEANED),
              'centerlines': str(NETWORK if CENTERLINE_KIND == 'network' else MAIN),
              'collapsed_links': str(CHILD_LINKS), 'collapse_edits': str(EDITS),
              'base_links': str(BASE_LINKS), 'membership': str(MEMBERSHIP),
              'locations': str(LOCATIONS_PATH), 'land': str(land_path), 'ocean': str(ocean_path),
              'unit_id': 35, 'crs': str(CRS), 'bounds': list(BOUNDS),
              'prepared_source': json.loads((RUN/'prepare_summary.json').read_text())['source_mask']}
if SAVE_PROVENANCE:
    (OUTPUT_DIR / 'figure_sources.json').write_text(json.dumps(provenance, indent=2))
display(pd.DataFrame({'input': ['source (Q4)', 'prepared (Q1)', 'cleaned (Q1)'],
                      'path': [str(SOURCE), str(PREPARED), str(CLEANED)]}))
print('Collapsed link IDs:', collapsed.id_link.tolist())


## 2.1 Original 2020Q4 water raster

In [ ]:
fig, ax = detail_panel('', source_image, binary=False)
finish(fig, '2_1_original_water_raster')

## 2.2 Prepared binary projected mask

In [ ]:
fig, ax = detail_panel('', prepared_image)
finish(fig, '2_2_prepared_mask')

## 2.3 Cleaned mask

In [ ]:
from scipy.ndimage import binary_fill_holes

ISLAND_COLOR = '#ffffff'
# Enclosed land pixels distinguish the island from banks connected to the crop edge.
cleaned_water = np.asarray(cleaned_image.filled(0)) > 0
island_pixels = binary_fill_holes(cleaned_water) & ~cleaned_water
fig, ax = detail_panel('', cleaned_image)
ax.imshow(np.ma.masked_where(~island_pixels, np.ones(cleaned_water.shape)),
          cmap=ListedColormap([ISLAND_COLOR]), vmin=0, vmax=1,
          extent=(BOUNDS[0], BOUNDS[2], BOUNDS[1], BOUNDS[3]),
          origin='upper', interpolation='nearest')
finish(fig, '2_3_cleaned_mask')

## 2.4 Cleaned mask with centerlines

In [ ]:
from shapely.geometry import Point, LineString, box
from matplotlib.path import Path as MarkerPath
from matplotlib.markers import MarkerStyle
from matplotlib.transforms import Affine2D

# Orient arrows using upstream/downstream nodes, not arbitrary coordinate order.
flow_nodes = gpd.read_file(
    BASE / 'variant/directed/05__S000_base_directed_nodes.gpkg'
).to_crs(CRS).set_index('id_node').geometry
flow_links = base_links.set_index('id_link')
ARROW_HEAD_STYLE = 'vee'  # 'triangle', 'chevron' (open), or 'vee' (notched)
ARROW_HEAD_SIZE = 15  # points; independent of map scale
DIRECTION_SAMPLE_M = 50  # distance on each side used to estimate the tangent

# All shapes have their tip at (0, 0), placing it at the visible midpoint.
HEAD_PATHS = {
    'triangle': MarkerPath([(0, 0), (-1, .5), (-1, -.5), (0, 0)],
                          [MarkerPath.MOVETO, MarkerPath.LINETO, MarkerPath.LINETO, MarkerPath.CLOSEPOLY]),
    'chevron': MarkerPath([(-1, .5), (0, 0), (-1, -.5)],
                         [MarkerPath.MOVETO, MarkerPath.LINETO, MarkerPath.LINETO]),
    'vee': MarkerPath([(0, 0), (-1, .5), (-.65, 0), (-1, -.5), (0, 0)],
                     [MarkerPath.MOVETO, MarkerPath.LINETO, MarkerPath.LINETO,
                      MarkerPath.LINETO, MarkerPath.CLOSEPOLY]),
}

def line_parts(geometry):
    if geometry.is_empty:
        return []
    if geometry.geom_type == 'LineString':
        return [geometry] if geometry.length > 0 else []
    return [line for part in getattr(geometry, 'geoms', []) for line in line_parts(part)]

def add_flow_arrows(ax):
    # Clip to the actual image window before measuring visible lengths.
    view = box(x0, y0, x1, y1)
    if CENTERLINE_KIND == 'network':
        arrow_lines = centerlines
    else:
        main_corridor = centerlines.geometry.union_all().buffer(pixel_size)
        arrow_lines = base_links.copy()
        arrow_lines.geometry = arrow_lines.geometry.intersection(main_corridor)
    placements = []
    for row in arrow_lines.itertuples():
        upstream = flow_nodes.loc[flow_links.loc[row.id_link, 'id_us_node']]
        reference = flow_links.loc[row.id_link].geometry
        if Point(reference.coords[0]).distance(upstream) > Point(reference.coords[-1]).distance(upstream):
            reference = LineString(list(reference.coords)[::-1])
        pieces = line_parts(row.geometry.intersection(view))
        # Keep fragments in downstream order if a vector exits and re-enters the image.
        pieces = [LineString(list(part.coords)[::-1])
                  if reference.project(Point(part.coords[0])) > reference.project(Point(part.coords[-1]))
                  else part for part in pieces]
        pieces.sort(key=lambda part: reference.project(Point(part.coords[0])))
        visible_length = sum(part.length for part in pieces)
        if not visible_length:
            continue
        distance = visible_length / 2
        for segment in pieces:
            if distance <= segment.length:
                break
            distance -= segment.length
        midpoint = segment.interpolate(distance)
        # Arrowhead sits at the visible arc-length midpoint, pointing downstream.
        # Estimate the local tangent without using off-image geometry.
        delta = min(DIRECTION_SAMPLE_M, segment.length / 4)
        before = segment.interpolate(max(0, distance - delta))
        after = segment.interpolate(min(segment.length, distance + delta))
        direction = np.array([after.x - before.x, after.y - before.y])
        angle = np.arctan2(direction[1], direction[0])
        marker = MarkerStyle(HEAD_PATHS[ARROW_HEAD_STYLE],
                             transform=Affine2D().rotate(angle))
        # A marker draws only the head: no annotation shaft is added.
        ax.scatter([midpoint.x], [midpoint.y], marker=marker, s=ARROW_HEAD_SIZE**2.7,
                   facecolors='red' if ARROW_HEAD_STYLE == 'chevron' else 'red',
                   edgecolors='white', linewidths=.8, zorder=4, clip_on=True)
        placements.append({'id_link': row.id_link, 'visible_length_m': visible_length,
                           'midpoint': midpoint})
    return placements

fig, ax = detail_panel('', cleaned_image)
centerlines.plot(ax=ax, color=NETWORK_COLOR, linewidth=4, zorder=3)
arrow_placements = add_flow_arrows(ax)
ax.set(xlim=(x0, x1), ylim=(y0, y1))
finish(fig, '2_4_centerlines')

## 2.5 Cleaned mask, centerlines, and collapsed unit 35

In [ ]:
fig, ax = detail_panel('', cleaned_image)
# Display-only buffer to cover the raster/polygon edge gap.
FILL_BUFFER_M = 0.5 * pixel_size
fills_display = fills.copy()
fills_display.geometry = fills_display.geometry.buffer(FILL_BUFFER_M)
fills_display.plot(ax=ax, facecolor='grey', edgecolor='none', alpha=1, zorder=2)
centerlines.plot(ax=ax, color=NETWORK_COLOR, linewidth=4, zorder=3)
collapsed.plot(ax=ax, color=COLLAPSED_COLOR, linewidth=3, linestyle='--', zorder=4)
ax.set(xlim=(x0, x1), ylim=(y0, y1))
        # ax.legend(handles=[Line2D([], [], color=NETWORK_COLOR, label='Original centerlines'),
        #                    Line2D([], [], color=COLLAPSED_COLOR, linewidth=2, label='Collapsed unit 35'),
        #                    Patch(facecolor='#edb96e', alpha=.55, label='Filled island')],
        #           loc='upper left', frameon=True, fontsize=8)
finish(fig, '2_5_collapsed')

# 3. Example unit networks

Three cleaned-mask views with the same colors, 2-point centerlines, equal map aspect, and 500 m scale bar as section 2.4, without arrows. Example 5 reuses the exact section 2 extent; the other extents use their selected links plus `PADDING_FRACTION`.

Example 2 “compound unit 5” selects **compound bubble 5** from `02_compound_bubbles__anchored` (`CB001_bubble_5`), including all its member units. It does not select ordinary unit 5 alone. All channel branches visible in each crop are plotted.

In [ ]:
def plot_unit_network(example_id, selection_id, compound=False, bounds=None, figure_name=None):
    example = f'{example_id:02d}'
    experiment_kind = 'compound_bubbles' if compound else 'indep'
    experiment = ROOT / f'network_variants/output/{example}_{experiment_kind}__anchored'
    base = experiment / 'states/S000_base'
    run = ROOT / f'rivgraph_centerline/outputs/runs/{example}'
    mask_path = run / f'masks_cleaned/{example}_cleaned.tif'
    network_path = run / f'rivgraph/{example}_links.gpkg'
    links_path = base / f'variant/directed/{example}__S000_base_directed_links.gpkg'
    membership_path = base / 'variant/summary/unit_link_membership.csv'
    members = pd.read_csv(membership_path)
    key = 'compound_bubble_id' if compound else 'unit_id'
    selected_members = members.loc[members[key].eq(selection_id)]
    selected_ids = selected_members.id_link.unique()
    if not len(selected_ids):
        raise ValueError(f'Example {example}: no links for {key}={selection_id}')
    with rasterio.open(mask_path) as src:
        crs, resolution = src.crs, abs(src.res[0])
    links = gpd.read_file(links_path).to_crs(crs)
    selected = links.loc[links.id_link.isin(selected_ids)]
    assert set(selected.id_link) == set(selected_ids), 'Missing selected link geometries'
    if bounds is None:
        left, bottom, right, top = selected.total_bounds
        padding = max(right-left, top-bottom) * PADDING_FRACTION
        bounds = (left-padding, bottom-padding, right+padding, top+padding)
    left, bottom, right, top = bounds
    cols = int(np.ceil((right-left) / resolution))
    rows = int(np.ceil((top-bottom) / resolution))
    transform = from_bounds(*bounds, cols, rows)
    with rasterio.open(mask_path) as src:
        with WarpedVRT(src, crs=crs, transform=transform, width=cols, height=rows,
                       resampling=Resampling.nearest) as vrt:
            mask = vrt.read(1, masked=True)
    assert mask.count() > 0, 'The selected crop has no valid mask pixels'
    network = gpd.read_file(network_path).to_crs(crs)
    label = 'compound unit' if compound else 'unit'
    fig, ax = detail_panel(f'Example {example_id} · {label} {selection_id}', mask, bounds=bounds,
                           scale_position='left' if compound else 'right')
    network.plot(ax=ax, color=NETWORK_COLOR, linewidth=2, zorder=3)
    ax.set(xlim=(left, right), ylim=(bottom, top))
    if figure_name is None:
        raise ValueError('Provide the section-based output filename.')
    name = str(figure_name)
    sources = {'example_id': example_id, 'selection_field': key, 'selection_id': selection_id,
               'member_unit_ids': sorted(map(int, selected_members.unit_id.unique())),
               'selected_link_ids': sorted(map(int, selected_ids)),
               'mask': str(mask_path), 'centerlines': str(network_path),
               'membership': str(membership_path), 'base_links': str(links_path),
               'bounds': list(map(float, bounds)), 'crs': str(crs), 'arrows': False}
    if SAVE_PROVENANCE:
        (OUTPUT_DIR / f'{name}_sources.json').write_text(json.dumps(sources, indent=2))
    finish(fig, name)
    return sources


def plot_synthetic_island_case(case):
    """Draw simplified island topologies using the Section 3 network style."""
    lw = 10  # Line width for the network lines     
    ds = 400 # size of bifurcation/confluence nodes
    fig, ax = plt.subplots(figsize=(7, 4.7))
    def line(points):
        ax.plot(*zip(*points), color=NETWORK_COLOR, linewidth=lw,
                solid_capstyle='round', solid_joinstyle='round', zorder=3)
    island_style = dict(facecolor='#BDBDBD', edgecolor='none', alpha=.95, zorder=1)
    if case == 'simple':
        ax.add_patch(Ellipse((10, 0), 7, 3.8, **island_style))
        line([(0, 0), (4, 0), (6.2, 3.7), (13.8, 3.7), (16, 0), (20, 0)])
        line([(4, 0), (6.2, -3.7), (13.8, -3.7), (16, 0)])
        ax.scatter([4, 16], [0, 0], s=ds, c='black', zorder=4)
    elif case == 'nested':
        # The smaller island sits within the upper branch of the larger one.
        # Both islands have a true divergence and convergence node.
        ax.add_patch(Ellipse((10, -.25), 7.0, 4.0, **island_style)) # big island

        ax.add_patch(Ellipse((10, 3.6), 2.4, .6, **island_style)) # small island
        line([(0, 0), (4, 0), (6.5, 3.6), (7.5, 3.6)])              # first top line segment

        line([(7.5, 3.6), (8.8, 4.8), (11.2, 4.8), (12.5, 3.6),     # top nested line segment
              (13.5, 3.6), (16, 0), (20, 0)])                       # top line continuation
        line([(7.5, 3.6), (8.8, 2.4), (11.2, 2.4), (12.5, 3.6)])    # bottom nested line segment
        
        line([(4, 0), (6.5, -3.7), (13.5, -3.7), (16, 0)])          # bottom line segment
        ax.scatter([4, 7.5, 12.5, 16], [0, 3.6, 3.6, 0],
                   s=ds, c='black', zorder=4)
    elif case == 'compound':
        ax.add_patch(Ellipse((7, 0), 4, 4.6, **island_style))
        ax.add_patch(Ellipse((13, 0), 4, 4.6, **island_style))
        # One cross-branch connector, as in the synthetic crossover network.
        # Its endpoints sit on the upper and lower branches and route through
        # the gap between the two island pockets.
        line([(0, 0), (4, 0), (5.6, 3.7), (9.25, 3.7),
              (14.4, 3.7), (16, 0), (20, 0)])
        line([(4, 0), (5.6, -3.7), (10.75, -3.7),
              (14.4, -3.7), (16, 0)])
        line([(9.25, 3.7), (10.75, -3.7)])
        ax.scatter([4, 9.25, 10.75, 16], [0, 3.7, -3.7, 0],
                   s=ds, c='black', zorder=4)
    else:
        raise ValueError(f'Unknown synthetic island case: {case}')
    ax.set(xlim=(-.5, 20.5), ylim=(-6, 6), aspect='equal')
    ax.set_axis_off()
    return fig


## 3.1 Simple unit

### 3.1.1 Real life case
Example 5 · unit 35

In [ ]:
example05_unit35_sources = plot_unit_network(5, 35, bounds=BOUNDS, figure_name='3_1_1_real')

### 3.1.2 Synthetic example

A simple unit with one idealised island. This schematic shows the intended topology rather than a scaled reconstruction of Example 5.

In [ ]:
synthetic_simple = plot_synthetic_island_case('simple')
finish(synthetic_simple, '3_1_2_synthetic')

## 3.2 Nested unit

### 3.2.1 Real life case
 Example 56 · unit 12

In [ ]:
example56_unit12_sources = plot_unit_network(56, 12, figure_name='3_2_1_real')

### 3.2.2 Synthetic example

A nested configuration: a large main island contains a smaller island within its internal braid.

In [ ]:
synthetic_nested = plot_synthetic_island_case('nested')
finish(synthetic_nested, '3_2_2_synthetic')

## 3.3 Compound unit

### 3.3.1 Real life case
Example 2 · compound unit 5

In [ ]:
example02_compound5_sources = plot_unit_network(2, 5, compound=True, figure_name='3_3_1_real')

### 3.3.2 Synthetic example

A compound configuration with one idealised island in each pocket.

In [ ]:
synthetic_compound = plot_synthetic_island_case('compound')
finish(synthetic_compound, '3_3_2_synthetic')

# 4. Width measurements — example 5, unit 35

Adapted from `RAPID/notebooks/width_investigation.ipynb`, **Paper figures for width investigation**, function `dt_view`. Original exports are in `/Users/6256481/Desktop/PhD_icloud/projecten/river_hierarchy/paper/figuren/methode/width_metric_links_…`.

The reference style is retained: grey bank components, white water, black target links, blue left-bank legs/targets, green right-bank legs/targets, and optional orange collapsed-island fill. No arrows, axes, title, or legend. All valid samples are shown (`step=1`), including valid samples excluded by later width trimming, matching the original function.

These figures use current width products (Parquet where available). Collapsed link IDs are selected by lineage rather than the old notebook's hard-coded link 102. Geometry/measurements may therefore differ from historical exports.

In [ ]:
from rasterio.features import shapes as raster_shapes
from rasterio.windows import from_bounds as window_from_bounds, Window
from shapely.geometry import box, shape

WIDTH_BANK_COLORS = ['#B0B0B0', '#3B3B3B', '#696969']
WIDTH_SAMPLE_STEP = 1

def width_asset(variant, key, folder, fallback=None):
    manifest = json.loads((variant / 'summary/variant_manifest.json').read_text())
    stored = manifest['output_paths'].get(key)
    candidates = []
    if stored:
        candidates.extend([variant / folder / Path(stored).name, Path(stored)])
    if fallback:
        candidates.append(variant / folder / fallback)
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'{key} missing in {variant}: {candidates}')

def width_view(variant, link_ids, name, secondary_links=None, show_collapse=False,
               pad_m=150, bounds=None, bank_colors=WIDTH_BANK_COLORS, step=WIDTH_SAMPLE_STEP,
               show_width_lines=True, show_bank_targets=True,
               centerline_color="black", context_color="gray"):
    variant = Path(variant)
    link_ids = [int(link_ids)] if np.isscalar(link_ids) else list(map(int, link_ids))
    links_path = width_asset(variant, 'directed_links', 'directed')
    labels_path = width_asset(variant, 'bank_labels', 'mask')
    sample_path = variant / 'widths/link_width_samples.parquet'
    if sample_path.exists():
        samples = pd.read_parquet(sample_path, filters=[('id_link', 'in', link_ids)])
    else:
        sample_path = width_asset(variant, 'link_width_samples', 'widths')
        samples = pd.read_csv(sample_path)
    links = gpd.read_file(links_path)
    target = links.loc[links.id_link.isin(link_ids)].copy()
    missing = sorted(set(link_ids) - set(target.id_link))
    if missing:
        raise ValueError(f'Missing link IDs: {missing}')
    selected = samples.loc[samples.id_link.isin(link_ids)].sort_values(['id_link', 'sample_index'])
    invalid = selected.width_invalid.fillna(False).astype(str).str.lower().isin(['true', '1'])
    valid = selected.loc[~invalid].iloc[::step].copy()
    coords = ['x', 'y', 'left_bank_x', 'left_bank_y', 'right_bank_x', 'right_bank_y']
    valid = valid.loc[np.isfinite(valid[coords]).all(axis=1)]
    if valid.empty:
        raise ValueError(f'No valid width samples for links {link_ids}')
    if bounds is None:
        xs = np.r_[target.total_bounds[[0, 2]], valid.x, valid.left_bank_x, valid.right_bank_x]
        ys = np.r_[target.total_bounds[[1, 3]], valid.y, valid.left_bank_y, valid.right_bank_y]
        bounds = (float(xs.min()-pad_m), float(ys.min()-pad_m),
                  float(xs.max()+pad_m), float(ys.max()+pad_m))
    aoi = box(*bounds)
    bank_ids = pd.concat([valid.left_component_id, valid.right_component_id]).dropna().astype(int).unique()
    polygons = []
    with rasterio.open(labels_path) as src:
        if src.crs != links.crs:
            raise ValueError('Bank labels and links must share a CRS')
        window = window_from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
        window = window.intersection(Window(0, 0, src.width, src.height))
        labels = src.read(1, window=window)
        transform = src.window_transform(window)
        for geometry, value in raster_shapes(labels.astype('int32'),
                                             mask=np.isin(labels, bank_ids), transform=transform):
            if int(value) > 1:
                polygons.append({'bank_id': int(value), 'geometry': shape(geometry)})
    banks = gpd.GeoDataFrame(polygons, columns=['bank_id', 'geometry'], geometry='geometry', crs=links.crs)
    fig, ax = plt.subplots(figsize=(10, 10))
    if not banks.empty:
        banks = banks.dissolve(by='bank_id').reset_index().clip(aoi)
        colors = [bank_colors[i % len(bank_colors)] for i in range(len(banks))]
        banks.plot(ax=ax, facecolor=colors, edgecolor='0.55', linewidth=1.2, zorder=1)
    edits_path = None
    if show_collapse:
        edits_path = width_asset(variant, 'edit_geometries', 'mask', 'collapse_edit_geometries.gpkg')
        edits = gpd.read_file(edits_path).to_crs(links.crs)
        island = edits.loc[edits.geometry_role.eq('edit') & edits.action.eq('add')].clip(aoi)
        if not island.empty:
            island.plot(ax=ax, facecolor='#FF9D00', edgecolor='darkgoldenrod', alpha=.35, zorder=1.5)
    target.plot(ax=ax, color=centerline_color, linewidth=2.5, zorder=3)
    if secondary_links is not None:
        context = links.loc[links.id_link.isin(secondary_links)]
        if not context.empty:
            context.plot(ax=ax, color=context_color, linewidth=2, zorder=3)
    if show_width_lines:
        for row in valid.itertuples():
            ax.plot([row.left_bank_x, row.x], [row.left_bank_y, row.y],
                    color='#0072B2', alpha=.5, linewidth=1, zorder=2)
            ax.plot([row.x, row.right_bank_x], [row.y, row.right_bank_y],
                    color='#009E73', alpha=.5, linewidth=1, zorder=2)
    if show_bank_targets:
        ax.scatter(valid.left_bank_x, valid.left_bank_y, s=16, color='#0072B2', zorder=4)
        ax.scatter(valid.right_bank_x, valid.right_bank_y, s=16, color='#009E73', zorder=4)
    ax.set(xlim=(bounds[0], bounds[2]), ylim=(bounds[1], bounds[3]), aspect='equal')
    ax.axis('off')
    sources = {'reference_notebook': 'RAPID/notebooks/width_investigation.ipynb',
               'variant': str(variant), 'links': str(links_path), 'samples': str(sample_path),
               'bank_labels': str(labels_path), 'edits': str(edits_path) if edits_path else None,
               'link_ids': link_ids, 'secondary_links': secondary_links,
               'bounds': list(map(float, bounds)), 'crs': str(links.crs),
               'valid_samples': len(valid), 'step': step, 'show_collapse': show_collapse,
               'show_width_lines': show_width_lines, 'show_bank_targets': show_bank_targets,
               'bank_colors': list(bank_colors), 'centerline_color': centerline_color,
               'context_color': context_color}
    if SAVE_PROVENANCE:
        (OUTPUT_DIR / f'{name}_sources.json').write_text(json.dumps(sources, indent=2))
    finish(fig, name)
    return sources

WIDTH_BASE05 = ROOT / 'network_variants/output/05_indep__anchored/states/S000_base/variant'
WIDTH_CHILD05 = ROOT / 'network_variants/output/05_indep__anchored/states/S035_unit_35/variant'
width_members05 = pd.read_csv(WIDTH_BASE05 / 'summary/unit_link_membership.csv')
width_unit35_ids = sorted(width_members05.loc[width_members05.unit_id.eq(35), 'id_link'].unique().tolist())
width_child05_links = gpd.read_file(width_asset(WIDTH_CHILD05, 'directed_links', 'directed'))
width_collapsed35_ids = width_child05_links.loc[
    width_child05_links.lineage_type.eq('collapsed_many_to_one'), 'id_link'].astype(int).tolist()
assert width_unit35_ids and width_collapsed35_ids


## 4.1 Original branches — example 5, unit 35

In [ ]:
width_unit35_base_sources = width_view(WIDTH_BASE05, width_unit35_ids, '4_1_orig')

## 4.2 Collapsed channel — example 5, unit 35

In [ ]:
width_unit35_child_sources = width_view(WIDTH_CHILD05, width_collapsed35_ids, '4_2_collapse', bank_colors=['#B0B0B0', '#696969', '#3B3B3B'])

## 4.3 Collapsed channel with island fill — example 5, unit 35

In [ ]:
width_unit35_fill_sources = width_view(WIDTH_CHILD05, width_collapsed35_ids, '4_3_collapse_filled', show_collapse=True, bank_colors=['#B0B0B0', '#696969', '#3B3B3B'])

# Results

# Appendix


## Width review

### Example 22 · link 530

Same reference styling, including neighbouring links 528, 531, 533, and 534 in grey, as in the original notebook.

In [ ]:
width_link530_sources = width_view(
    ROOT / 'network_variants/output/22_indep__anchored/states/S000_base/variant',
    [530], 'A_1_width_review_angle', secondary_links=[528, 531, 533, 534])

### Example 5 · compound unit 33 — wider local view

Selects compound bubble 33 (base links 4–8), with 300 m padding instead of the original 150 m. This is a local crop of example 5; change `pad_m` or pass projected `bounds=(left, bottom, right, top)` to adjust the partial view.

Simplified review view: uniform grey banks and black centerlines, without cross-sectional lines or bank-target dots.

In [ ]:
width_compound05 = ROOT / 'network_variants/output/05_compound_bubbles__anchored/states/S000_base/variant'
width_compound_members = pd.read_csv(width_compound05 / 'summary/unit_link_membership.csv')
width_compound33_ids = sorted(width_compound_members.loc[
    width_compound_members.compound_bubble_id.eq(33), 'id_link'].unique().tolist())
assert width_compound33_ids
width_compound33_links = gpd.read_file(width_asset(width_compound05, 'directed_links', 'directed'))
width_compound33_context = sorted(set(width_compound33_links.id_link.astype(int)) - set(width_compound33_ids))
width_compound33_sources = width_view(
    width_compound05, width_compound33_ids, 'A_1_width_review_reason',
    secondary_links=width_compound33_context, pad_m=300,
    bank_colors=['#B0B0B0'], centerline_color='black', context_color='black',
    show_width_lines=False, show_bank_targets=False)